# Run GeneID

In [ ]:
#geneid_command="time geneid -3P {parameter_file} {assembly} > {annotation.gff3}"

tr_sp=!ls ../data/species
#tr_sp=["Plasmodium_vivax"]

for src in ["own", "git"]:
    #generate 1 file per source
    with open(f"../job_commands/SARpred_{src}.txt", "w") as out:
        for sp in tr_sp:
            
            #reference and model name are adaptable to the source
            ref=!realpath ../data/species/$sp/CLEAN*fna 
            model=f"../results/{sp}/{sp}_{src}.gff3"

            #parameter location changes depending on source
            param=!realpath ../results/$sp/*param
            param=param[0]
            #use github parameters
            if src=="git":
                filename_git=f"../data/geneid-parameter-files/parameter_files/{sp}*.param"

                try: #if no github parameters exist, dont runn geneID for git parameters

                    found_files=!realpath $filename_git 2>/dev/null #supress error output
                    #set up error for species with no github parameters
                    if "*" in found_files[0]: #found files is exactly filename_git if nothing is found
                        raise FileNotFoundError(f"There is no github parameters for this species")
                    
                except Exception as e:
                    print(f"{sp} presents error: {e}")
                    continue

                param=found_files[0]

            #geneid param_own/param_git fasta_assembly > gff_model
            command = f"time geneid -3P {param} {ref[0]} > {model}"
            print(command)
            temporrrious=f"{param} {ref[0]} > {model}"
            out.write(command + "\n")

#Execute commands

['/home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Brachionus_asplanchnoidis/Brachionus_asplanchnoidis.geneid.optimized.param', '/home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Brachionus_asplanchnoidis/Brachionus_asplanchnoidis.geneid.param']
time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Brachionus_asplanchnoidis/Brachionus_asplanchnoidis.geneid.param /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Brachionus_asplanchnoidis/CLEAN_Basplanchnoidis_gn.fna > ../results/Brachionus_asplanchnoidis/Brachionus_asplanchnoidis_own.gff3
['/home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Cloeon_dipterum/Cloeon_dipterum.geneid.optimized.param', '/home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Cloeon_dipterum/Cloeon_dipterum.geneid.param']
time geneid -3P /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Cloeon_dipterum/Cl

# Prediction analysis

## Reference annotation(CDS) and Prediction comparison

In [ ]:
tr_sp=!ls ../data/species
#tr_sp=["Mytilus_galloprovincialis"]

for src in ["own", "git"]:
    with open(f"../job_commands/gffcomp_{src}.txt", "w") as f:
        for sp in tr_sp:
        #for sp in ["Drosophila_melanogaster"]:
            try:
                #if no github parameters exist, dont process this species git comparison
                filename_git=f"../data/geneid-parameter-files/parameter_files/{sp}*.param"
                found_files=!realpath $filename_git 2>/dev/null #supress error output

                if not found_files:
                    #@remove created file?? #@correct path errors
                    raise FileNotFoundError(f"There is no github parameters for this species")

                #reference annotation
                RefAnn=!realpath ../data/species/$sp/CDS_*gff
                RefAnn=RefAnn[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/$sp/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store comparison information
                resloc=f"../results/{sp}/gffcmp"
                folder_cmd=f"mkdir -p {resloc}"

                #comparison command
                comparing_cmd=f"gffcompare -r {RefAnn} {pred} -o {resloc}/GFF_{src}"

                #move residual files generatd by gffcompare into the folder as well
                extrafiles=f"../results/{sp}/GFF_{src}"
                moving_cmd=f"mv {extrafiles}* {resloc}"
                
                print(folder_cmd, file=f)
                print(comparing_cmd, file=f)
                print(comparing_cmd)
                print(moving_cmd, file=f)
                print(f'echo "done_{sp}"', file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue

gffcompare -r /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Plasmodium_vivax/CDS_Pvivax_ann.gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Plasmodium_vivax/Plasmodium_vivax_own.gff3 -o ../results/Plasmodium_vivax/gffcmp/GFF_own
gffcompare -r /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Plasmodium_vivax/CDS_Pvivax_ann.gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/results/Plasmodium_vivax/Plasmodium_vivax_git.gff3 -o ../results/Plasmodium_vivax/gffcmp/GFF_git


### Result comparison

#### .stats files present info

**According to previous statistics**

- git and own have pretty similar results in both species, but in Pvivax they are closer(more similar) than in Pfalc. We can see this in the probability spreads, both are more disperse in Pfalc
- Individual distributuins show slight skew toward own data while transition matrix probabilities have a more stable distribution


**Both are confirmed by looking at the .stats files of the geneID results:**

- Git and Own results are pretty similar, specially in Pvivax
- Own results are slightly better


**Summary:** own produces better results

## BUSCO assessment

In [ ]:
#agat for gff>prot
tr_sp=!ls ../data/species

#for src in ["git", "own"]:
for src in ["own"]:
    with open(f"../job_commands/protExtract_{src}.txt", "w") as f:
        for sp in tr_sp:
            try:
                #if no github parameters exist, dont process this species git comparison
                filename_git=f"../data/geneid-parameter-files/parameter_files/{sp}*.param"
                found_files=!realpath $filename_git 2>/dev/null #supress error output

                if not found_files:
                    #@remove created file?? #@correct path errors
                    raise FileNotFoundError(f"There is no github parameters for this species")
                
                #reference sequence
                RefAnn=!realpath ../data/species/$sp/*fa
                RefAnn=RefAnn[0]

                #annotation prediction by source (own or git)
                pred=!realpath ../results/$sp/$sp*_$src*.gff3
                pred=pred[0]

                #create folder to store comparison information
                resloc=f"../results/{sp}/gffcmp"
                folder_cmd=f"mkdir -p {resloc}"

                #comparison command
                comparing_cmd=f"gffcompare -r {RefAnn} {pred} -o {resloc}/GFF_{src}"

                #move residual files generatd by gffcompare into the folder as well
                extrafiles=f"../results/{sp}/GFF_{src}"
                moving_cmd=f"mv {extrafiles}* {resloc}"
                
                print(folder_cmd, file=f)
                print(comparing_cmd, file=f)
                print(comparing_cmd)
                print(moving_cmd, file=f)
                print(f'echo "done_{sp}"', file=f)

            except Exception as e:
                print(f"{sp} presents error: {e}")
                continue
            


agat_sp_extract_sequences.pl \
    -f "$FASTA_FILE" \
    -g "$LONGISOFORM_GFF" \
    -t CDS \
    -p \
    -o "$PROTEIN_OUTPUT"

busco -m protein \
	-i ${proteins} \
	-l ${lineage} \
	--cpu 4 \
    --offline \
	--out_path "${OUT_PATH}" \
	-o "${OUT_NAME}"

# Setup for IGV transcript comparison

In [ ]:
#generate symlinks to a igv allfiles folder
#s_results/specie/gff_*.gtf
#s_results/specie/gff_*.stats
#s_results/specie/*.gff3

    
#symlink files
#for raw files
clean_seq=!realpath ../data/species/**/CLEAN*.fna
clean_ann=!realpath ../data/species/**/CDS*ann.gff
rawData=clean_seq+clean_ann

for file in rawData:
    sp=file.split("/")[-2]
    #remove species shortened name from filename
    purename=file.split("/")[-1].split("_")
    purename="_".join([purename[0], purename[-1]])
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vs $file $dest/og_$purename


#for prediction result files
gff3=!realpath ../results/**/*.gff3

for file in gff3:
    sp=file.split("/")[-2]
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vs $file $dest


#for gffcompare files
gtfComp=!realpath ../results/**/gffcmp/*.gtf
statsComp=!realpath ../results/**/gffcmp/*.stats
gffComp=gtfComp+statsComp

for file in gffComp:
    sp=file.split("/")[-3]
    #name cleanup
    purename=file.split("/")[-1]
    purename=purename.replace(".annotated", "_ann").replace("GFF", "prediction")
    dest=f"../sum_results/{sp}"
    !mkdir -p $dest
    !ln -vs $file $dest/$purename






'../nres/Plasmodium_falciparum/og_CLEAN_gn.fna' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_falciparum/CLEAN_Pfalci_gn.fna'
'../nres/Plasmodium_vivax/og_CLEAN_gn.fna' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_vivax/CLEAN_Pvivax_gn.fna'
'../nres/Plasmodium_falciparum/og_CDS_ann.gff' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_falciparum/CDS_Pfalci_ann.gff'
'../nres/Plasmodium_vivax/og_CDS_ann.gff' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/data/species/Plasmodium_vivax/CDS_Pvivax_ann.gff'
'../nres/Plasmodium_falciparum/Plasmodium_falciparum_git.gff3' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/results/Plasmodium_falciparum/Plasmodium_falciparum_git.gff3'
'../nres/Plasmodium_falciparum/Plasmodium_falciparum_own.gff3' -> '/home/jj/Desktop/Data_science/CRG/TFM/geneid-bg/results/Plasmodium_falciparum/Plasmodium_falciparum_own.gff3'
'../nres/Plasmodium_vivax/Plasmodium_vivax_git